In [155]:
# Imports and definition of paths

import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob
from pandas.io.parquet import to_parquet

RAW_DIR = Path('../data/raw/sinca')

In [156]:
# Load file per station
csv_files = sorted(RAW_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(' ', f.name)

Found 11 CSV files:
  Cerrillos II.csv
  Cerro Navia.csv
  El Bosque.csv
  Independencia.csv
  La Florida.csv
  Las Condes.csv
  Parque O'Higgins.csv
  Pudahuel.csv
  Puente Alto.csv
  Quilicura.csv
  Talagante.csv


In [157]:
# Load and concatenate all files
frames = []
for fp in csv_files:
    try:
        df_tmp = pd.read_csv(fp, sep=';', encoding='latin-1', low_memory=False)
    except Excfeption:
        df_tmp = pd.read_csv(fp, sep=',', encoding='latin-1', low_memory=False)
   
    df_tmp['source_file'] = fp.name
    frames.append(df_tmp)

air = pd.concat(frames, ignore_index=True)
print(air.shape)
print(air.dtypes)
air.head()

(500032, 7)
FECHA (YYMMDD)              int64
HORA (HHMM)                 int64
Registros validados       float64
Registros preliminares    float64
Registros no validados    float64
Unnamed: 5                float64
source_file                   str
dtype: object


,FECHA (YYMMDD),HORA (HHMM),Registros validados,Registros preliminares,Registros no validados,Unnamed: 5,source_file
0,220401,0,NaN,NaN,NaN,NaN,Cerrillos II.csv
1,220402,0,NaN,NaN,NaN,NaN,Cerrillos II.csv
2,220403,0,NaN,NaN,NaN,NaN,Cerrillos II.csv
3,220404,0,NaN,NaN,NaN,NaN,Cerrillos II.csv
4,220405,0,NaN,NaN,NaN,NaN,Cerrillos II.csv


In [158]:
# Rename column names

column_mapping = {
    'FECHA (YYMMDD)': 'date',
    'HORA (HHMM)': 'hour',
    'Registros validados': 'pm25',
    'source_file': 'station'
}

air = air.rename(columns= column_mapping)

# Drop columns we do not need
air = air.drop(columns= ['Registros preliminares', 'Registros no validados', 'Unnamed: 5'])

print(air.columns.tolist())
print(air.shape)

['date', 'hour', 'pm25', 'station']
(500032, 4)


In [159]:
# Parse column date
air['date'] = pd.to_datetime(
    air['date'].astype(str).str.zfill(6),
    format='%y%m%d', 
    errors='coerce'
)

failed = air['date'].isna().sum()
print(f'Rows that failed to parse: {failed}')

print('Date range:', air['date'].min(), 'to', air['date'].max())

Rows that failed to parse: 0
Date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00


In [160]:
# Parse column hour
air['hour'] = (air['hour'] // 100).astype(int)

In [161]:
# Fix mixed values and missing data in PM2.5 column

air['pm25'] = (
    air['pm25']
        .astype(str)
        .str.replace(',', '.', regex= False)
        .pipe(pd.to_numeric, errors= 'coerce')
)

# Replace SINCA missing values sentinel
air['pm25']  = air['pm25'].replace(-999, np.nan)

# Remove physically impossible values
air.loc[air['pm25'] > 500, PM25_COL] = np.nan

# Summary
total = len(air)
missing = air['pm25'].isna().sum()
print(f'Total rows: {total}')
print(f'Missing PM2.5 values: {missing} ({missing/total*100:.1f}%)')

Total rows: 500032
Missing PM2.5 values: 44590 (8.9%)


In [189]:
# Summary
print(f'Total rows   : {len(air)}')
print(f'\nDate range   : {air["date"].min()} to {air["date"].max()}')
print(f'\nMissing PM2.5: {air["pm25"].isna().sum()} ({air["pm25"].isna().mean()*100:.1f}%)')
print(f'\nColumns      : {air.columns.tolist()}')
print(f'\nStatios in dataset: ')
for station in sorted(air['station'].unique()):
    count = air[air['station'] == station].shape[0]
    print(f'     {station}:   {count} records')

Total rows   : 500032

Date range   : 2020-01-01 00:00:00 to 2025-12-31 00:00:00

Missing PM2.5: 44590 (8.9%)

Columns      : ['date', 'hour', 'pm25', 'station', 'Registros validados']

Statios in dataset: 
     Cerrillos II.csv:   1370 records
     Cerro Navia.csv:   52607 records
     El Bosque.csv:   52607 records
     Independencia.csv:   25199 records
     La Florida.csv:   52607 records
     Las Condes.csv:   52607 records
     Parque O'Higgins.csv:   52607 records
     Pudahuel.csv:   52607 records
     Puente Alto.csv:   52607 records
     Quilicura.csv:   52607 records
     Talagante.csv:   52607 records


In [190]:
INTERIM_DIR = Path('../data/interim')
INTERIM_DIR.mkdir(exist_ok=True)

df_raw.to_parquet(INTERIM_DIR / 'air_quality_raw.parquet', index=False)
print('Saved successfully')

Saved successfully
